# Defining variables

In [0]:
catalog = "workspace"
schema = "test_schema"
volume = "test_vol"
download_url = "https://health.data.ny.gov/api/views/jxy9-yhdk/rows.csv"
file_name = "rows.csv"
table_name = "nyx_table"
path_volume = "/Volumes/"+ catalog + "/" + schema + "/" + volume
path_table = catalog + "." + schema
print(path_volume)
print(path_table)

# loading CSV file

In [0]:
dbutils.fs.cp(f"{download_url}", f"{path_volume}/{file_name}")
# it seems the free version of Databricks doesn't allow to download from a url,
# so, I am going to download the datamanually from the url, save it and then upload it to the test_vol


# Creating a test DataFrame
This method is useful when creating small test data sets, unit testing transformations, ​prototyping logic.


In [0]:
data = [[2021, "test", "Albany", "M", 42]]
columns = ["Year", "First_Name", "County", "Sex","Count"]

df1 = spark.createDataFrame(data, schema="Year int, First_Name STRING, County STRING, Sex STRING, Count int")
df1.display()

# Load the downloaded CSV data into DataFrame

In [0]:
df_csv = spark.read.csv(f"{path_volume}/{file_name}",
                          header= True,
                          inferSchema=True,
                          sep=",")

df_csv.display()


# Verify and Interact with DataFrame

## Print DataFrame schema

In [0]:
df_csv.printSchema()
df1.printSchema()

## create some other DataFrame operations

In [0]:
#df_csv.count()
#df_csv.columns
#df_csv.describe().show()
display(df_csv.summary())

# Combine DataFrame

In [0]:
df = df1.union(df_csv)
display(df)
df.printSchema()

# filter rows in dataframe

In [0]:
# using .filter() method
display(df.filter(df["Count"]>50))

In [0]:
# Using .where() method
display(df.where(df["Count"]>50))

# select columns from a dataframe and order by freqeuncy

In [0]:
from pyspark.sql.functions import desc

display(df.select("First_Name", "Count").orderBy(desc("Count")))

# create a subset of the dataframes

In [0]:
subsetDF = df.filter((df["Year"]==2009)& 
                     (df["Count"]>100) &
                     (df["Sex"] == "F")
                     ).select("First_Name", "County", "Count").orderBy(desc("Count"))

display(subsetDF)

# Rename a column

In [0]:
subsetDF = subsetDF.withColumnRenamed("Count", "Total")

subsetDF.display()

# Save the dataframe to a multiple location

In [0]:
# save the dataframe into a table

df.write.mode("overwrite").saveAsTable(f"{path_table}.{table_name}")

In [0]:
# save into JSON file format

#df.write.format("json").mode("overwrite").save("/tmp/json_data") # this is probably gonna work in the paid version of Databricks.

display(df.write.format("json").mode("overwrite").save("/Volumes/workspace/test_schema/test_vol/json_data"))


In [0]:
display(spark.read.format("json").json("/Volumes/workspace/test_schema/test_vol/json_data"))

In [0]:
display(spark.sql(f"SELECT * FROM {path_table}.{table_name}"))